# EverFlavor AI - Extra Data Collection

**Team:** EverGlow
**Course:** ITAI 2277
**Project:** EverFlavor AI - Multi-Agent Recipe and Nutrition System

---

This notebook collects extra recipe data to fill the gaps found in notebook 01, mainly the small number of African and Middle Eastern recipes. Each source is saved to `data/raw/` as a Parquet file. Week 5 of notebook 01 loads these files automatically if they exist.

| Source | What it adds | Access |
|---|---|---|
| CulinaryDB | 45,772 recipes from 22 world regions, including 651 African and 993 Middle Eastern (titles and ingredient lists) | Free download, CC BY-NC-SA 3.0 |
| TheMealDB | About 800 recipes from many countries, each with instructions and ingredients | Free public API (test key) |
| RecipeDB (optional) | About 118,000 recipes with nutrition values | API key from the CoSyLab team |

We only use official downloads and APIs. We do not scrape recipe websites directly, because their terms of use forbid it and their page layouts change often.

**Run order:** run this notebook any time before Week 5 of notebook 01. It works in Google Colab, VS Code and Antigravity.

---

## 1. Environment Setup

The same setup as notebook 01: it detects where the notebook is running, moves to the project folder, loads secrets from Colab Secrets or `config/.env`, and creates the data folders. Run these two cells first, and again after every kernel restart.

In [1]:
# %pip (not !pip) installs into the Python that runs this notebook,
# in Colab and in local kernels (VS Code / Antigravity / JupyterLab).
%pip install -q requests pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import importlib.util
from pathlib import Path

# -------------------------------------------------------
# 1. Where is this notebook running?
# -------------------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUNTIME = "Google Colab" if IN_COLAB else "Local Jupyter (VS Code / Antigravity / JupyterLab)"

# -------------------------------------------------------
# 2. Move to the project folder so every relative path
#    (data/raw, data/processed, config/.env) means the same thing
#    everywhere. Local kernels usually start inside
#    notebooks/, so we walk up to the folder that holds
#    both notebooks/ and data/.
# -------------------------------------------------------
def find_project_root(start):
    """Return the nearest folder at or above `start` with notebooks/ and data/, or None."""
    start = Path(start).resolve()
    for folder in [start, *start.parents]:
        if (folder / "notebooks").is_dir() and (folder / "data").is_dir():
            return folder
    return None

PROJECT_ROOT = find_project_root(Path.cwd())
if PROJECT_ROOT is None and IN_COLAB and Path("/content/The-EverFlavor-AI").is_dir():
    # The repo was cloned into Colab with: !git clone <repo url>
    PROJECT_ROOT = Path("/content/The-EverFlavor-AI")
if PROJECT_ROOT is None:
    # Notebook opened on its own (e.g. Colab straight from GitHub): work in the current folder
    PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)


def short_path(p):
    """Show a path with the home folder as ~, so usernames stay out of saved outputs."""
    p, home = str(p), str(Path.home())
    return "~" + p[len(home):] if p.startswith(home) else p

# -------------------------------------------------------
# 3. Secrets.
# Local runs: read KEY=value lines from config/.env in the
# project folder (.env is in .gitignore, so it is never pushed).
# Values already set in the environment are not replaced,
# and empty values are skipped.
# -------------------------------------------------------
def load_env_file(env_path):
    """Load KEY=value lines from an .env file into os.environ. Returns True if found."""
    if not env_path.is_file():
        return False
    # utf-8-sig also handles files saved by Windows Notepad (which adds a BOM)
    with open(env_path, encoding="utf-8-sig") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            if key.startswith("export "):
                key = key[len("export "):].strip()
            value = value.strip().strip('"').strip("'")
            if value and key not in os.environ:
                os.environ[key] = value
    return True

ENV_FILE_FOUND = load_env_file(PROJECT_ROOT / "config" / ".env")


def get_secret(name):
    """Return a secret by name, or None if it is not set anywhere.

    Order: Colab Secrets (in Colab), then environment variables,
    which include everything loaded from .env.
    """
    if IN_COLAB:
        try:
            from google.colab import userdata
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            # Secret not added, or notebook access not switched on
            pass
    return os.environ.get(name) or None

# Libraries such as kagglehub and Hugging Face read these from
# environment variables, so copy them there if they are set.
for _name in ["HF_TOKEN", "KAGGLE_USERNAME", "KAGGLE_KEY"]:
    if _name not in os.environ:
        _value = get_secret(_name)
        if _value:
            os.environ[_name] = _value

# -------------------------------------------------------
# 4. Create the standard project folder structure
# -------------------------------------------------------
folders = ["data/raw", "data/interim", "data/processed"]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# -------------------------------------------------------
# 5. Report (secret names only, never their values)
# -------------------------------------------------------
REQUIRED_PACKAGES = {"requests": "requests", "pandas": "pandas", "pyarrow": "pyarrow"}
missing_packages = [pkg for pkg, module in REQUIRED_PACKAGES.items()
                    if importlib.util.find_spec(module) is None]

print(f"Runtime        : {RUNTIME}")
print(f"Python         : {sys.version.split()[0]}")
print(f"Project folder : {short_path(PROJECT_ROOT)}")
if not IN_COLAB:
    print(f".env file      : {'found' if ENV_FILE_FOUND else 'not found (copy config/.env.example to config/.env)'}")
print(f"  THEMEALDB_API_KEY: {'set' if get_secret('THEMEALDB_API_KEY') else 'not set (the free test key is used)'}")
for folder in folders:
    print(f"Folder ready   : {folder}")

if sys.version_info < (3, 10):
    print("\nWARNING: Python 3.10 or newer is required.")
if missing_packages:
    print(f"\nWARNING: missing packages: {', '.join(missing_packages)}")
    print("Run the %pip install cell above, then restart the kernel.")

print("\nEnvironment is set up.")

Runtime        : Local Jupyter (VS Code / Antigravity / JupyterLab)
Python         : 3.12.10
Project folder : ~\Downloads\jupiter-exploration\The-EverFlavor-AI
.env file      : found
  THEMEALDB_API_KEY: not set (the free test key is used)
Folder ready   : data/raw
Folder ready   : data/interim
Folder ready   : data/processed

Environment is set up.


## 2. CulinaryDB

**Source:** https://cosylab.iiitd.edu.in/culinarydb/
**License:** Creative Commons Attribution-NonCommercial-ShareAlike 3.0 (credit the Complex Systems Lab, IIIT-Delhi; non-commercial use only)

CulinaryDB is a free download (about 5 MB zipped). It has recipe titles, a regional cuisine and an ingredient list for each recipe, but no instructions or nutrition values. We use it mainly for its African and Middle Eastern recipes.

In [3]:
import io
import time
import zipfile
import requests
import pandas as pd

CULINARYDB_URL = "https://cosylab.iiitd.edu.in/culinarydb/static/data/CulinaryDB.zip"
CULINARYDB_DIR = Path("data/raw/culinarydb")
CULINARYDB_OUT = "data/raw/culinarydb_recipes.parquet"


def find_file(folder, name):
    """Return the first file called `name` anywhere under `folder`, or None."""
    return next(Path(folder).rglob(name), None)

# Download and unzip once; later runs reuse the files
if find_file(CULINARYDB_DIR, "01_Recipe_Details.csv") is None:
    print("Downloading CulinaryDB ...")
    response = requests.get(CULINARYDB_URL, timeout=120)
    response.raise_for_status()
    zipfile.ZipFile(io.BytesIO(response.content)).extractall(CULINARYDB_DIR)
else:
    print("CulinaryDB already downloaded.")

details = pd.read_csv(find_file(CULINARYDB_DIR, "01_Recipe_Details.csv"))
aliases = pd.read_csv(find_file(CULINARYDB_DIR, "04_Recipe-Ingredients_Aliases.csv"))

# One row per recipe, with its ingredients as a list
ingredient_lists = (aliases.dropna(subset=["Original Ingredient Name"])
                    .groupby("Recipe ID")["Original Ingredient Name"].apply(list))
culinarydb = pd.DataFrame({
    "recipe_id"  : details["Recipe ID"],
    "recipe_name": details["Title"],
    "cuisine"    : details["Cuisine"],
    "origin_site": details["Source"],   # where CulinaryDB got the recipe
    "ingredients": details["Recipe ID"].map(ingredient_lists),
})
culinarydb = culinarydb.dropna(subset=["recipe_name", "ingredients"])

culinarydb.to_parquet(CULINARYDB_OUT, index=False)
print(f"Saved {len(culinarydb):,} recipes to {CULINARYDB_OUT}")
print("\nRecipes by region:")
print(culinarydb["cuisine"].value_counts().to_string())

Saved 45,749 recipes to data/raw/culinarydb_recipes.parquet

Recipes by region:
cuisine
USA                       16106
Italy                      7502
Indian Subcontinent        4057
Mexico                     3138
France                     2701
Canada                     1112
Caribbean                  1103
British Isles              1074
Middle East                 993
China                       941
Greece                      934
Spain                       815
Thailand                    666
Africa                      650
South East Asia             611
Japan                       578
Eastern Europe              565
Australia & NZ              494
DACH Countries              487
Scandinavia                 404
South America               310
Korea                       301
Misc.: Portugal             138
Misc.: Dutch                 40
Misc.: Belgian               15
Misc.: Central America       14


## 3. TheMealDB

**Source:** https://www.themealdb.com/api.php
**Access:** free public API. The test key `1` is meant for development and education, which fits this project. If the team gets a supporter key, add it as `THEMEALDB_API_KEY` (Colab Secrets or `config/.env`).

TheMealDB is small but complete: every recipe has instructions, ingredients with measures, a category and a country. We fetch every recipe by searching each first letter and digit (36 requests), with a short pause between requests.

In [4]:
import string

MEALDB_KEY = get_secret("THEMEALDB_API_KEY") or "1"   # "1" is the free test key
MEALDB_URL = f"https://www.themealdb.com/api/json/v1/{MEALDB_KEY}/search.php"
MEALDB_OUT = "data/raw/themealdb_recipes.parquet"

# Status codes that mean "try again later" (rate limit or server busy)
RETRY_STATUS = {429, 500, 502, 503, 504}


def get_json(url, params=None, retries=3, backoff=2.0):
    """GET a URL and return its JSON, retrying temporary errors with a growing wait."""
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, params=params, timeout=20)
        except (requests.ConnectionError, requests.Timeout):
            if attempt == retries:
                raise
            time.sleep(backoff * attempt)
            continue
        if response.status_code in RETRY_STATUS and attempt < retries:
            time.sleep(backoff * attempt)
            continue
        if not response.ok:
            # Do not print the URL: it contains the API key
            raise RuntimeError(f"TheMealDB request failed with HTTP {response.status_code}")
        return response.json()


def meal_to_row(meal):
    """Flatten one TheMealDB record (ingredients are in 20 numbered fields)."""
    ingredients, measures = [], []
    for i in range(1, 21):
        name = (meal.get(f"strIngredient{i}") or "").strip()
        if name:
            ingredients.append(name)
            measures.append((meal.get(f"strMeasure{i}") or "").strip())
    return {
        "recipe_id"   : meal["idMeal"],
        "recipe_name" : meal.get("strMeal"),
        # Newer records use strCountry instead of strArea
        "area"        : meal.get("strArea") or meal.get("strCountry"),
        "category"    : meal.get("strCategory"),
        "instructions": meal.get("strInstructions"),
        "ingredients" : ingredients,
        "measures"    : measures,
        "tags"        : meal.get("strTags"),
        "source_url"  : meal.get("strSource"),
    }

meals = {}
for first_char in string.ascii_lowercase + string.digits:
    try:
        data = get_json(MEALDB_URL, params={"f": first_char})
    except Exception as e:
        # Only the error type: the full message can contain the URL with the key
        print(f"  Warning: search for '{first_char}' failed ({type(e).__name__})")
        continue
    for meal in data.get("meals") or []:
        meals[meal["idMeal"]] = meal
    time.sleep(0.3)   # be polite to the free API

themealdb = pd.DataFrame([meal_to_row(m) for m in meals.values()])
if themealdb.empty:
    print("No recipes retrieved. Check your network connection and run this cell again.")
else:
    themealdb.to_parquet(MEALDB_OUT, index=False)
    print(f"Saved {len(themealdb):,} recipes to {MEALDB_OUT}")
    print("\nRecipes by country (top 25):")
    print(themealdb["area"].fillna("(none)").value_counts().head(25).to_string())

Saved 791 recipes to data/raw/themealdb_recipes.parquet

Recipes by country (top 25):
area
British          59
Spanish          48
United States    34
Turkish          30
France           28
Chinese          27
Vietnamese       27
Polish           27
Jamaican         27
Thai             27
Canadian         22
Italian          21
Netherlands      18
Norway           17
Dominica         16
Cambodia         15
India            15
Colombia         13
Australian       13
Algerian         12
Saudi Arabian    12
Brazil           11
Albania          11
Argentina        10
Venezuela        10


## 4. RecipeDB (Optional)

**Source:** https://cosylab.iiitd.edu.in/recipedb/
**License:** CC BY-NC-SA 3.0

RecipeDB has about 118,000 recipes from 26 geo-cultural regions, with nutrition values. It would help most with African and Middle Eastern coverage. Its API needs a key from the CoSyLab team at IIIT-Delhi; request one through their website and mention that this is a student project.

Once the team has a key, store it as `RECIPEDB_API_KEY` in Colab Secrets or `config/.env`, and add a collection cell here following the same pattern as TheMealDB (`get_secret()`, `get_json()` with retries, save to `data/raw/recipedb_recipes.parquet`).

## 5. Summary

In [5]:
summary = []
for name, file_path in {"CulinaryDB": CULINARYDB_OUT, "TheMealDB": MEALDB_OUT}.items():
    if os.path.exists(file_path):
        rows = len(pd.read_parquet(file_path))
        summary.append((name, f"{rows:,}", file_path, f"{os.path.getsize(file_path) / 1e6:.1f} MB"))
    else:
        summary.append((name, "0", file_path, "not saved"))

print(pd.DataFrame(summary, columns=["source", "recipes", "file", "size"]).to_string(index=False))
print("\nNext: run Week 5 of notebooks/01_data_acquisition_EverFlavor_V3.ipynb; it loads these files automatically.")

    source recipes                                file   size
CulinaryDB  45,749 data/raw/culinarydb_recipes.parquet 7.1 MB
 TheMealDB     791  data/raw/themealdb_recipes.parquet 0.5 MB

Next: run Week 5 of notebooks/01_data_acquisition_EverFlavor_V3.ipynb; it loads these files automatically.


## 6. Credits

- **CulinaryDB:** Complex Systems Lab (CoSyLab), IIIT-Delhi. https://cosylab.iiitd.edu.in/culinarydb/ (CC BY-NC-SA 3.0)
- **TheMealDB:** https://www.themealdb.com

These sources are used for a non-commercial student project. Check their licenses again before any commercial use.